# 3. Generación de embeddings

Este notebook es el **tercer paso del pipeline**, donde se generan los embeddings de las noticias con **Qwen3-Embedding-0.6B**.


## 0. Configuración y dependencias

In [ ]:
!pip install sentence-transformers umap-learn hdbscan optuna scikit-learn pandas pyarrow seaborn

In [2]:
import re
import warnings
import html
import os
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import gc

warnings.filterwarnings('ignore')


In [3]:
PROC_DIR   = Path('../data/processed')
EMB_DIR    = Path('../embeddings')
OUTPUT_DIR = Path('../output')
for d in [PROC_DIR, EMB_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MEDIOS = [
    'elconfidencial',
    'eldiario',
    'publico',
    'elmundo',
    'okdiario',
    'elpais',
]

print(f'Medios: {MEDIOS}')


Medios: ['elconfidencial', 'eldiario', 'publico', 'elmundo', 'okdiario', 'elpais']


## 1. Carga de datos

Se carga el corpus ya procesado y filtrado por `1_limpieza.ipynb` y `2_eda.ipynb`. Se construye la columna `texto` concatenando titular, abstract y noticia.


In [4]:
def limpiar(texto):
    if not isinstance(texto, str):
        return ''
    texto = html.unescape(texto)
    texto = texto.replace('\\n', ' ').replace('\\t', ' ').replace('\\"', '"').replace('\\r', ' ')
    texto = re.sub(r'<[^>]+>', ' ', texto)
    texto = re.sub(r'http\S+', '', texto)
    texto = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', texto)
    texto = re.sub(r'\s\w{1,3}[_]$', '', texto)
    texto = re.sub(r'\.\s?\.\s?\.', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto


# Cargar corpus filtrado (generado por 1_limpieza + 2_eda)
df = pd.read_parquet(PROC_DIR / 'corpus_limpio_filtrado.parquet')
df['fecha_dt'] = pd.to_datetime(df['fecha'], errors='coerce')

# Normalizar nombres de medios a IDs cortos
mapping = {
    'El Confidencial': 'elconfidencial',
    'elDiario.es':     'eldiario',
    'Público':         'publico',
    'El Mundo':        'elmundo',
    'OKDiario':        'okdiario',
    'El País':         'elpais',
}
df['medio'] = df['medio'].map(mapping).fillna(df['medio'])

# Construir texto completo para el embedding: titular + abstract + noticia
df['texto'] = (
    df['titular'].fillna('') + '. ' +
    df['abstract'].fillna('') + '. ' +
    df['noticia'].fillna('')
).apply(limpiar)
df['len_texto'] = df['texto'].str.len()

print(f'Total artículos: {len(df)}')
print(df.groupby('medio').size().to_string())
print(f"\nLongitud media : {df['len_texto'].mean():.0f} caracteres")
print(f"Longitud máxima: {df['len_texto'].max()} | mínima: {df['len_texto'].min()}")
print(f"\nMedios en df   : {sorted(df['medio'].unique())}")
print(f"Medios esperados: {sorted(MEDIOS)}")


Total artículos: 241240
medio
elconfidencial    36718
eldiario          38494
elmundo           23012
elpais            64936
okdiario          52023
publico           26057

Longitud media : 4668 caracteres
Longitud máxima: 296086 | mínima: 144

Medios en df   : ['elconfidencial', 'eldiario', 'elmundo', 'elpais', 'okdiario', 'publico']
Medios esperados: ['elconfidencial', 'eldiario', 'elmundo', 'elpais', 'okdiario', 'publico']


## 2. Modelo de embeddings

Se utiliza **Qwen3-Embedding-0.6B** cargado con `SentenceTransformer`. El modelo se ubica automáticamente en el dispositivo disponible: MPS (Apple Silicon), CUDA (GPU NVIDIA) o CPU.


In [5]:
# Dispositivo
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Dispositivo: {device}")

Dispositivo: mps


In [6]:
# Modelo
MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'
model = SentenceTransformer(
    MODEL_NAME,
    device=str(device),
    trust_remote_code=True,
    model_kwargs={'torch_dtype': torch.float16}
)
model.eval()
print(f'Modelo cargado: {MODEL_NAME}')
print(f'Modelo en: {next(model.parameters()).device}')

# Verificar que modelo y parámetros están en el mismo dispositivo
print(f'Parámetros en: {next(model.parameters()).device}')


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Modelo cargado: Qwen/Qwen3-Embedding-0.6B
Modelo en: mps:0
Parámetros en: mps:0


In [7]:
# Análisis de tokens (sobre todos los medios)
tokenizer = model.tokenizer
sample = tokenizer(df["texto"].tolist()[:100], truncation=False, return_length=True)
lengths = sample["length"]
print(f"\nAnálisis tokens (muestra 100):")
print(f"  Media  : {sum(lengths)/len(lengths):.0f} tokens")
print(f"  Máximo : {max(lengths)} tokens")


Análisis tokens (muestra 100):
  Media  : 1312 tokens
  Máximo : 3647 tokens


## 3. Generación de embeddings

La función `procesar_embeddings` genera los embeddings por batches con tres mecanismos de robustez:
- **Checkpoints parciales** cada 500 artículos: permite retomar si la ejecución se interrumpe.
- **Reducción automática de batch** ante OOM en MPS: divide el batch a la mitad hasta batch=1.
- **Guardado por medio**: si el `.npy` ya existe en disco, se salta ese medio — es seguro re-ejecutar.


In [8]:
def procesar_embeddings(lista_textos, batch_size=4, checkpoint_path=None, save_every=500):
    print('Usando funcion nueva con try/except')
    import numpy as np
    from pathlib import Path

    def is_oom(e):
        msg = str(e).lower()
        return 'out of memory' in msg or 'invalid buffer size' in msg

    # Retomar desde checkpoint
    all_embeddings = []
    skipped = []
    start_i = 0
    if checkpoint_path and Path(checkpoint_path).exists():
        saved = np.load(checkpoint_path)
        all_embeddings.append(saved)
        start_i = saved.shape[0]
        print(f'Retomando desde artículo {start_i} (checkpoint cargado)')

    emb_dim = None
    i = start_i
    with tqdm(total=len(lista_textos), initial=start_i, desc='Generando') as pbar:
        while i < len(lista_textos):
            batch = lista_textos[i:i + batch_size]
            try:
                emb = model.encode(
                    batch,
                    normalize_embeddings=True,
                    convert_to_numpy=True,
                    show_progress_bar=False
                )
                if emb_dim is None:
                    emb_dim = emb.shape[1]
                all_embeddings.append(emb)
                i += batch_size
                pbar.update(len(batch))
                torch.mps.empty_cache()

                # ── Checkpoint parcial cada save_every artículos ──
                if checkpoint_path and (i % save_every < batch_size):
                    np.save(checkpoint_path, np.vstack(all_embeddings))

            except RuntimeError as e:
                if is_oom(e):
                    torch.mps.empty_cache()
                    gc.collect()
                    if batch_size > 1:
                        batch_size = max(1, batch_size // 2)
                        print(f'[WARN] OOM -> reduciendo batch_size a {batch_size}')
                    else:
                        # batch_size=1 y sigue fallando -> saltar directamente
                        dim = emb_dim if emb_dim else model.get_sentence_embedding_dimension()
                        all_embeddings.append(np.zeros((1, dim), dtype=np.float32))
                        skipped.append({'indice': i, 'texto_inicio': lista_textos[i][:100], 'error': str(e)[:120]})
                        print(f'[SKIP] articulo {i} imposible en MPS -> vector cero ({len(lista_textos[i])} chars)')
                        i += 1
                        pbar.update(1)
                        torch.mps.empty_cache()
                else:
                    raise e

    # ── Resumen de artículos saltados ─────────────────────────────
    if skipped:
        import json as _json
        skip_path = (Path(checkpoint_path).parent / 'skipped_articulos.json') if checkpoint_path else Path(
            '../embeddings/skipped_articulos.json')
        skip_path.write_text(_json.dumps(skipped, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f'\n[RESUMEN] {len(skipped)} artículos saltados -> {skip_path}')
        for s in skipped:
            print(f'  idx={s["indice"]}  chars={len(lista_textos[s["indice"]])}  texto: {s["texto_inicio"]}')
    else:
        print('\n[RESUMEN] Ningún artículo saltado.')

    return np.vstack(all_embeddings)


In [9]:
if str(device) == 'mps':
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'
    torch.mps.empty_cache()
gc.collect()

emb_por_medio = {}

for medio in MEDIOS:
    embs_path    = EMB_DIR / f'{medio}_embeddings.npy'
    partial_path = EMB_DIR / f'{medio}_embeddings_partial.npy'

    if embs_path.exists():
        emb_por_medio[medio] = np.load(embs_path)
        print(f'[{medio}] cargado desde disco → shape: {emb_por_medio[medio].shape}')
        continue

    textos = df[df['medio'] == medio]['texto'].tolist()
    print(f'\n[{medio}] {len(textos)} artículos...')

    embeddings = procesar_embeddings(
        textos,
        batch_size=4,
        checkpoint_path=str(partial_path),
        save_every=500,
    )
    np.save(embs_path, embeddings)
    # Borrar partial ahora que tenemos el final
    if partial_path.exists():
        partial_path.unlink()
    emb_por_medio[medio] = embeddings
    print(f'[{medio}] guardado → shape: {embeddings.shape}')


[elconfidencial] cargado desde disco → shape: (37178, 1024)
[eldiario] cargado desde disco → shape: (38637, 1024)
[publico] cargado desde disco → shape: (26602, 1024)
[elmundo] cargado desde disco → shape: (23012, 1024)
[okdiario] cargado desde disco → shape: (52023, 1024)
[elpais] cargado desde disco → shape: (64936, 1024)


Finalmente, se han generado los embeddings de los **6 medios** con **Qwen3-Embedding-0.6B** (dimensión 1 024):

| Medio | Artículos | Fichero |
|-------|----------:|---------|
| El Confidencial | 37 178 | `elconfidencial_embeddings.npy` |
| elDiario.es | 38 637 | `eldiario_embeddings.npy` |
| Público | 26 602 | `publico_embeddings.npy` |
| El Mundo | 23 012 | `elmundo_embeddings.npy` |
| OKDiario | 52 023 | `okdiario_embeddings.npy` |
| El País | 64 936 | `elpais_embeddings.npy` |
| **Total** | **242 388** | `embeddings/` |

Cada fichero `.npy` contiene un array de forma `(n_artículos, 1024)` con embeddings L2-normalizados, listos para su uso.
